# Weather Forecasting using Machine Learning
### A complete, honest walkthrough — from raw data to a working forecast app

**The problem:** given everything we know about the weather up to today, predict
(1) tomorrow's mean temperature, (2) the next 7 days of temperature, and
(3) whether it will rain tomorrow — for 5 Indian cities: Indore, Delhi, Mumbai,
Bengaluru and Kolkata.

**Data:** the free Open-Meteo Historical (archive) API — no key needed. We use
2014-01-01 to the present (~12.8 years, 4,676 days per city, 23,380 rows total).
Hourly values are aggregated to daily: max/min/mean temperature, mean humidity,
mean pressure, max/mean wind speed, total precipitation, mean cloud cover.
The download is cached in `data/` so the project also works offline.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # project root, if run from notebooks/
if not os.path.exists(os.path.join("..", "src")):
    sys.path.insert(0, os.path.abspath("."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import clean_daily, load_daily

daily = clean_daily(load_daily())
print(daily.shape)
daily.head()

## 1. A first look at the data

Notice each row is ONE day in ONE city. Temperatures in India swing a lot over
a year (Delhi winters vs summers), so any good model must understand both
*recent weather* (yesterday was hot, today is probably hot) and *seasonality*
(June is hotter than January).

In [ ]:
indore = daily[daily.city == "Indore"].set_index("date")
plt.figure(figsize=(12, 3.5))
plt.plot(indore.index, indore.temp_mean, lw=0.6, alpha=0.6, label="Daily mean")
plt.plot(indore.index, indore.temp_mean.rolling(30).mean(), lw=1.8, label="30-day average")
plt.title("Indore — mean temperature, 2014 to today")
plt.ylabel("°C"); plt.legend(); plt.show()
print(daily.groupby("city").temp_mean.describe()[["mean", "min", "max"]])

## 2. Feature engineering — and how we avoid data leakage

We reframe the data as a **forecast-origin** problem: each training row stands
at the end of a day (the *origin*) and may use ONLY information available at
that moment. The target is the temperature `horizon` days later.

Features (all computed from data on/before the origin):
- **Lag features:** temperature 1, 2, 3, 7, 14 days before the forecast day
- **Rolling statistics:** mean & std of temperature over the last 3, 7, 30 days
- **Other sensors, latest values:** max/min temperature, humidity, pressure,
  wind, precipitation, cloud cover
- **Temperature range** (max − min) and **pressure change** over 1 and 3 days
- **Seasonality:** sin/cos of the target day's day-of-year (calendar info is
  known in advance, so it cannot leak)

**Leakage traps we avoided:** rolling windows END at the origin (never after),
the climatology baseline is computed from training years only, scaling happens
inside the sklearn Pipeline (fitted on train folds only), and model selection
uses TimeSeriesSplit cross-validation — random CV would let the model peek at
the future.

In [ ]:
from src.features import FEATURE_COLUMNS, build_supervised, split_time_based

sup = build_supervised(daily, horizon=1)
print("Supervised rows:", len(sup))
sup[["date", "target_date", "city", "temp_mean_lag1", "temp_roll_mean_7",
     "doy_sin", "target_temp"]].head()

## 3. Train / test split — time-based, no shuffling

Weather is a time series: shuffling would mix future into training. We hold out
the **latest 365 days** as the test set and train on everything before.
Hyperparameters are tuned with 3-fold **TimeSeriesSplit** on the training part,
and the winning model is chosen by cross-validation score — *before* anyone
looks at the test set.

In [ ]:
train, test, cutoff = split_time_based(sup, test_days=365)
print(f"Train: {len(train)} rows | Test: {len(test)} rows | cutoff: {cutoff.date()}")
X_COLS = FEATURE_COLUMNS + ["city"]

## 4. Baselines first — the models every ML method must beat

1. **Persistence:** tomorrow's temperature = today's. Embarrassingly strong
   for next-day forecasts.
2. **Seasonal average (climatology):** the average temperature for that
   calendar day, computed from training years only.

In [ ]:
from sklearn.metrics import mean_absolute_error
from src.features import climatology_table, seasonal_baseline_predict

clim = climatology_table(daily[daily.date <= cutoff])
pred_persist = test["temp_mean_lag1"]
pred_seasonal = seasonal_baseline_predict(clim, test["city"], test["target_date"])
print("Persistence  MAE:", round(mean_absolute_error(test.target_temp, pred_persist), 3))
print("Seasonal avg MAE:", round(mean_absolute_error(test.target_temp, pred_seasonal), 3))

## 5. Machine learning models

`train.py` tunes Ridge, Random Forest, XGBoost and an MLP with TimeSeriesSplit
grid search. Here we train two of them quickly to see the pattern (full results
are in `reports/metrics.json`).

In [ ]:
from src.models import get_regressors
from sklearn.metrics import mean_squared_error, r2_score

def report(name, model):
    model.fit(train[X_COLS], train.target_temp)
    p = model.predict(test[X_COLS])
    mae = mean_absolute_error(test.target_temp, p)
    rmse = np.sqrt(mean_squared_error(test.target_temp, p))
    print(f"{name:14s} MAE {mae:.3f} | RMSE {rmse:.3f} | R2 {r2_score(test.target_temp, p):.3f}")
    return model

regs = get_regressors(FEATURE_COLUMNS)
ridge = report("Ridge", regs["Ridge"][0])
xgb   = report("XGBoost", regs["XGBoost"][0])

**What actually happened in the full run:** on cross-validation the simple
**Ridge regression won** (CV MAE 0.626 °C) — with strong lag/rolling features,
next-day temperature is close to a linear problem, and the fancy models could
not beat it honestly. On the test year all ML models cluster tightly
(MAE 0.66–0.68 °C) and all beat persistence (0.702 °C) and crush the seasonal
average (1.464 °C). Lesson: *features + honest evaluation matter more than
model complexity.*

## 6. Multi-horizon forecasting — error grows with distance

For the 7-day outlook we train a **separate direct model per horizon**
(h = 1..7) instead of feeding predictions back recursively (which compounds
errors). As expected, MAE grows from ~0.67 °C at day 1 to ~1.54 °C at day 7 —
still far better than the seasonal baseline at every horizon.

In [ ]:
rows = []
for h in range(1, 8):
    sup_h = build_supervised(daily, horizon=h)
    tr_h, te_h, _ = split_time_based(sup_h, test_days=365)
    m = get_regressors(FEATURE_COLUMNS)["Ridge"][0]
    m.fit(tr_h[X_COLS], tr_h.target_temp)
    rows.append({"horizon": h, "MAE": round(mean_absolute_error(te_h.target_temp, m.predict(te_h[X_COLS])), 3)})
pd.DataFrame(rows)

## 7. Rain tomorrow — a classification problem with imbalanced classes

Only about a third of days have ≥ 1 mm of rain, and in dry months far fewer —
so accuracy alone is meaningless (predicting "no rain" always scores ~65-90%).
We use class-weighted models and judge by **precision, recall, F1 and ROC-AUC**.
In the full run a Random Forest won: **F1 0.812, ROC-AUC 0.943** on the test year.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from src.models import make_preprocessor
from sklearn.pipeline import Pipeline

rain_tr = train.dropna(subset=["target_rain"]); rain_te = test.dropna(subset=["target_rain"])
clf = Pipeline([("prep", make_preprocessor(FEATURE_COLUMNS)),
                ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
                                                 random_state=42, n_jobs=-1))])
clf.fit(rain_tr[X_COLS], rain_tr.target_rain.astype(int))
print(classification_report(rain_te.target_rain.astype(int), clf.predict(rain_te[X_COLS]),
                            target_names=["No rain", "Rain"]))

## 8. What drives the predictions?

Permutation importance (shuffle one feature, watch the error grow) shows the
model leans on the most recent temperatures — `temp_mean_lag1`, `temp_mean_lag2`,
the 3-day rolling mean — plus today's min/max. The SHAP summary in
`reports/figures/shap_summary.png` tells the same story per-prediction.

**Error analysis:** the model struggles most in June / the monsoon, when
temperature depends on rain and cloud events that yesterday's data cannot fully
reveal, and it is most accurate in calm November weather. Prediction intervals
(±~1 °C at 80%) are calibrated on a held-out slice of training data, so the app
can honestly say "26 °C, likely between 25 and 27 °C".

## 9. Run the real thing

- `python train.py` — retrains everything (about 15 min, mostly Random Forest tuning)
- `streamlit run app/app.py` — the demo app with live Open-Meteo forecasts